# Segmented6 Hybrid Retry Notebook

This notebook retries only the missing or failed segmented6 MobileNetV3Small hybrid runs.

- It does not rerun all 18 runs
- It does not delete successful outputs
- It does not start retry automatically


In [1]:
import gc
import importlib

import pandas as pd

import mobilenetv3small_segmented6_hybrid_lib as segmented6_lib

segmented6_lib = importlib.reload(segmented6_lib)

for _name in dir(segmented6_lib):
    if not _name.startswith("_"):
        globals()[_name] = getattr(segmented6_lib, _name)

print("Library source:", segmented6_lib.__file__)
print("Model input mode:", MODEL_INPUT_MODE)
print("Image crop mode:", IMAGE_CROP_MODE)
print("Folds:", EXTENSION_FOLDS)
print("Seeds:", EXTENSION_RUN_SEEDS)


[INFO] TensorFlow XLA/JIT disabled.
[INFO] TensorFlow XLA/JIT disabled.
Library source: e:\Thesis Code\mobilenetv3small_segmented6_hybrid_lib.py
Model input mode: cnn_plus_color_texture
Image crop mode: preprocessed_segmented_roi_224
Folds: ['fold1', 'fold2', 'fold3', 'fold4', 'fold5', 'fold6']
Seeds: [42, 123, 2026]


In [2]:
FAILED_RUNS_TO_RETRY = [
    ("fold3", 42),
    ("fold3", 123),
    ("fold3", 2026),
    ("fold4", 123),
    ("fold4", 2026),
    ("fold5", 123),
    ("fold5", 2026),
]

FAILED_RUNS_TO_RETRY


[('fold3', 42),
 ('fold3', 123),
 ('fold3', 2026),
 ('fold4', 123),
 ('fold4', 2026),
 ('fold5', 123),
 ('fold5', 2026)]

In [3]:
def retry_failed_segmented6_runs(failed_runs_to_retry):
    ensure_output_dirs()
    split_dfs = load_all_cross_rotation_splits()
    existing_metrics_df = load_existing_metrics()

    completed = []
    skipped = []
    failed = []

    for fold_name, seed in failed_runs_to_retry:
        run_stem = make_run_stem(fold_name, seed)
        model_path = EXTENSION_MODELS_ROOT / f"{run_stem}.h5"
        prediction_path = EXTENSION_PREDICTIONS_ROOT / f"{run_stem}_predictions.csv"

        already_in_metrics = False
        if existing_metrics_df is not None and not existing_metrics_df.empty:
            mask = (
                existing_metrics_df["fold_name"].astype(str).eq(str(fold_name)) &
                existing_metrics_df["seed"].astype(str).eq(str(seed)) &
                existing_metrics_df["model_input_mode"].astype(str).eq(MODEL_INPUT_MODE) &
                existing_metrics_df["image_crop_mode"].astype(str).eq(IMAGE_CROP_MODE)
            )
            already_in_metrics = bool(mask.any())

        if already_in_metrics and model_path.exists() and prediction_path.exists():
            print(f"[SKIP] {run_stem} already completed.")
            skipped.append((fold_name, seed))
            continue

        print(f"[RETRY] Running {run_stem}")

        try:
            result = run_single_segmented6_hybrid_experiment(
                fold_name=fold_name,
                seed=seed,
                split_dfs=split_dfs,
            )

            result_df = pd.DataFrame([result])
            if SEGMENTED6_SEED_METRICS_PATH.exists():
                old_df = pd.read_csv(SEGMENTED6_SEED_METRICS_PATH)
                old_df = old_df[
                    ~(
                        old_df["fold_name"].astype(str).eq(str(fold_name)) &
                        old_df["seed"].astype(str).eq(str(seed)) &
                        old_df["model_input_mode"].astype(str).eq(MODEL_INPUT_MODE) &
                        old_df["image_crop_mode"].astype(str).eq(IMAGE_CROP_MODE)
                    )
                ]
                new_df = pd.concat([old_df, result_df], ignore_index=True)
            else:
                new_df = result_df

            new_df.to_csv(SEGMENTED6_SEED_METRICS_PATH, index=False)
            existing_metrics_df = new_df.copy()

            print(f"[DONE] {run_stem}")
            completed.append((fold_name, seed))

        except Exception as exc:
            print(f"[FAILED] {run_stem}: {repr(exc)}")
            append_failed_run(fold_name, seed, exc)
            failed.append((fold_name, seed, repr(exc)))

            try:
                tf.keras.backend.clear_session()
            except Exception:
                pass
            gc.collect()

    print("Retry complete.")
    print("Completed:", completed)
    print("Skipped:", skipped)
    print("Failed:", failed)

    return {
        "completed": completed,
        "skipped": skipped,
        "failed": failed,
    }


In [5]:
MANUAL_CONFIRM_RETRY_FAILED_RUNS = True

if MANUAL_CONFIRM_RETRY_FAILED_RUNS:
    retry_result = retry_failed_segmented6_runs(FAILED_RUNS_TO_RETRY)
else:
    print("Failed-run retry is ready but not started.")
    print("Set MANUAL_CONFIRM_RETRY_FAILED_RUNS = True to retry only missing failed runs.")


[RETRY] Running segmented6_hybrid_fold3_seed42
[MEMORY] fold3 seed=42
  train_images count=2508 loaded_per_batch=32
  val_images count=442 loaded_per_batch=32
  test_images count=518 loaded_per_batch=32
  train_features shape=(2508, 30) approx_mb=0.29
  val_features shape=(442, 30) approx_mb=0.05
  test_features shape=(518, 30) approx_mb=0.06
[INFO] Using tf.keras.optimizers.legacy.Adam for DirectML compatibility.
[INFO] Compiled optimizer: keras.optimizers.legacy.adam.Adam
Epoch 1/8
 - val_f1_macro: 0.9416
79/79 - 29s - loss: 0.7066 - accuracy: 0.7037 - val_loss: 0.2618 - val_accuracy: 0.9412 - val_f1_macro: 0.9416 - lr: 5.0000e-04 - 29s/epoch - 368ms/step
Epoch 2/8
 - val_f1_macro: 0.9285
79/79 - 23s - loss: 0.3097 - accuracy: 0.8876 - val_loss: 0.2111 - val_accuracy: 0.9276 - val_f1_macro: 0.9285 - lr: 5.0000e-04 - 23s/epoch - 290ms/step
Epoch 3/8
 - val_f1_macro: 0.9352

Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
79/79 - 23s - loss: 0.2143 - accurac

INFO:tensorflow:Assets written to: C:\Users\jazre\AppData\Local\Temp\tmp96fjltbz\assets


INFO:tensorflow:Assets written to: C:\Users\jazre\AppData\Local\Temp\tmp96fjltbz\assets


[DONE] segmented6_hybrid_fold3_seed42
[RETRY] Running segmented6_hybrid_fold3_seed123
[MEMORY] fold3 seed=123
  train_images count=2508 loaded_per_batch=32
  val_images count=442 loaded_per_batch=32
  test_images count=518 loaded_per_batch=32
  train_features shape=(2508, 30) approx_mb=0.29
  val_features shape=(442, 30) approx_mb=0.05
  test_features shape=(518, 30) approx_mb=0.06
[INFO] Using tf.keras.optimizers.legacy.Adam for DirectML compatibility.
[INFO] Compiled optimizer: keras.optimizers.legacy.adam.Adam
Epoch 1/8
 - val_f1_macro: 0.9060
79/79 - 15s - loss: 0.6826 - accuracy: 0.7153 - val_loss: 0.2851 - val_accuracy: 0.9050 - val_f1_macro: 0.9060 - lr: 5.0000e-04 - 15s/epoch - 191ms/step
Epoch 2/8
 - val_f1_macro: 0.9441
79/79 - 10s - loss: 0.2912 - accuracy: 0.8931 - val_loss: 0.1931 - val_accuracy: 0.9434 - val_f1_macro: 0.9441 - lr: 5.0000e-04 - 10s/epoch - 131ms/step
Epoch 3/8
 - val_f1_macro: 0.9508
79/79 - 10s - loss: 0.2355 - accuracy: 0.9159 - val_loss: 0.1587 - val_ac

INFO:tensorflow:Assets written to: C:\Users\jazre\AppData\Local\Temp\tmplbkd123w\assets


...

INFO:tensorflow:Assets written to: C:\Users\jazre\AppData\Local\Temp\tmpe8ycu1f6\assets


[DONE] segmented6_hybrid_fold5_seed123
[RETRY] Running segmented6_hybrid_fold5_seed2026
[MEMORY] fold5 seed=2026
  train_images count=2438 loaded_per_batch=32
  val_images count=430 loaded_per_batch=32
  test_images count=600 loaded_per_batch=32
  train_features shape=(2438, 30) approx_mb=0.28
  val_features shape=(430, 30) approx_mb=0.05
  test_features shape=(600, 30) approx_mb=0.07
[INFO] Using tf.keras.optimizers.legacy.Adam for DirectML compatibility.
[INFO] Compiled optimizer: keras.optimizers.legacy.adam.Adam
Epoch 1/8
 - val_f1_macro: 0.9351
77/77 - 27s - loss: 0.7261 - accuracy: 0.6776 - val_loss: 0.2816 - val_accuracy: 0.9326 - val_f1_macro: 0.9351 - lr: 5.0000e-04 - 27s/epoch - 348ms/step
Epoch 2/8
 - val_f1_macro: 0.9257
77/77 - 20s - loss: 0.3277 - accuracy: 0.8774 - val_loss: 0.2204 - val_accuracy: 0.9233 - val_f1_macro: 0.9257 - lr: 5.0000e-04 - 20s/epoch - 263ms/step
Epoch 3/8
 - val_f1_macro: 0.9412
77/77 - 20s - loss: 0.2349 - accuracy: 0.9102 - val_loss: 0.1636 - val

INFO:tensorflow:Assets written to: C:\Users\jazre\AppData\Local\Temp\tmplgeh_fkd\assets


INFO:tensorflow:Assets written to: C:\Users\jazre\AppData\Local\Temp\tmplgeh_fkd\assets


[DONE] segmented6_hybrid_fold5_seed2026
Retry complete.
Completed: [('fold3', 42), ('fold3', 123), ('fold3', 2026), ('fold4', 123), ('fold4', 2026), ('fold5', 123), ('fold5', 2026)]
Skipped: []
Failed: []


In [6]:
MANUAL_CONFIRM_REGENERATE_AFTER_RETRY = True

if MANUAL_CONFIRM_REGENERATE_AFTER_RETRY:
    regenerate_transition_metrics_and_graphs()
else:
    print("Post-retry summary regeneration is ready but not started.")


Best run by strict macro F1: segmented6_hybrid_fold5_seed42 | macro F1=0.6433 | accuracy=0.6867
Mean strict macro F1 across all folds/seeds: 0.4349
Mean strict accuracy across all folds/seeds: 0.5125
Mean top-2 accuracy: 0.8693
Mean adjacent accuracy: 0.8904
Mean severe error rate: 0.1096
Mean ordinal error: 0.5971
Borderline rate: 0.4975
Most difficult true class based on recall/F1: not fresh
Most errors are adjacent rather than severe.
Strict accuracy and macro F1 are retained as the primary metrics. Transition-aware metrics are reported only as secondary analysis because pork freshness changes gradually and borderline images may visually lie between adjacent classes.


In [7]:
expected_runs = [(fold, seed) for fold in EXTENSION_FOLDS for seed in EXTENSION_RUN_SEEDS]

seed_metrics_df = pd.read_csv(SEGMENTED6_SEED_METRICS_PATH) if SEGMENTED6_SEED_METRICS_PATH.exists() else pd.DataFrame()

completed_pairs = set(
    zip(
        seed_metrics_df["fold_name"].astype(str),
        seed_metrics_df["seed"].astype(int),
    )
) if not seed_metrics_df.empty else set()

missing_pairs = [pair for pair in expected_runs if pair not in completed_pairs]

print("Expected runs:", len(expected_runs))
print("Completed runs:", len(completed_pairs))
print("Missing runs:", missing_pairs)

if len(missing_pairs) == 0:
    print("All 18 segmented6 hybrid runs are complete.")
else:
    print("Some runs are still missing. Retry only the missing runs.")


Expected runs: 18
Completed runs: 18
Missing runs: []
All 18 segmented6 hybrid runs are complete.
